# Module 5 — Fine-Tuning RemoteCLIP for Patch-Level Raster Classification

**Goal:** adapt RemoteCLIP (ViT-B/32) so each of the 49 patch tokens of an image (7x7 grid, 224px input, 32px patches) is classified into the raster vocabulary, using the manually verified labels from Module 4 as ground truth. This replaces the noisy zero-shot patch labeling (~30-40% correct) with a supervised model that can later run on the edge device.

**Method**

| Piece | Choice |
|---|---|
| Patch features | full ViT forward -> drop CLS token -> `ln_post` -> `proj` -> L2-normalise (512-d per patch) |
| Classifier | cosine similarity to **frozen** text prototypes (4-prompt ensemble per label) x fixed logit scale |
| Trainable weights | last `NUM_TRAINABLE_BLOCKS` transformer blocks of the **visual** tower only |
| Loss | cross-entropy, class-weighted by `1/sqrt(frequency)` |
| Model selection | best validation macro-F1 |
| Final numbers | RSICD's official **test** split |

**Protocol:** official RSICD train/val/test splits (restricted to the 8 labeled scene classes); label vocabulary built from **train** labels only; patches whose label never appears in train get target `-1` and are ignored.

**Inputs:** RSICD images + `dataset_rsicd.json` (official splits), verified raster JSONs (one per scene class).

**Outputs** (in `/kaggle/working/ERIC_Module5/`): `label_vocab.json`, `config.json`, `baseline_metrics.json`, `training_history.json`, `remoteclip_last2blocks_best.pt`, `test_metrics.json`, `per_class_report.json`.

> Run top to bottom (Restart & Run All). Hyper-parameters live in the Configuration cell only.

In [ ]:
!pip install -q open-clip-torch huggingface_hub scikit-learn

## 1. Configuration

All hyper-parameters and switches in one place.

- `LOGIT_SCALE` — multiplier applied to cosine similarities before the softmax. CLIP itself uses ~100; 10 is a soft setting (see *Known limitations* at the end).
- `DROP_UNCORRECTED_PLACEHOLDERS` — Module 3B fills images that failed all retries with one repeated placeholder label and flags `needs_review=True`. A grid that is **flagged and uniform** is almost certainly such an uncorrected placeholder, so it is dropped from training. Flagged grids with real variation (i.e. corrected in Module 4) are kept.

In [ ]:
SEED = 42

BATCH_SIZE = 32
NUM_WORKERS = 2

EPOCHS = 25
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0

NUM_TRAINABLE_BLOCKS = 2     # last N visual transformer blocks are unfrozen
LOGIT_SCALE = 10.0           # fixed scale on cosine similarities

DROP_UNCORRECTED_PLACEHOLDERS = True

assert NUM_TRAINABLE_BLOCKS >= 1, "NUM_TRAINABLE_BLOCKS must be >= 1 (a slice of [-0:] would unfreeze every block)"

## 2. Imports, Seeds and Device

In [ ]:
import os
import json
import glob
import random
import time
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import open_clip
from huggingface_hub import hf_hub_download
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

## 3. Paths

> **`QWEN_DIR` must contain the VERIFIED label files** (the Module 4 output, e.g. `airport_corrected.json`) — not the raw Qwen output from Module 3B, and not both (duplicates raise an error below). Otherwise the manual verification work never reaches training.

`GLOBAL_VOCAB_FILE` and `CLASS_TO_VOCAB_FILE` are only existence-checked here; they are not used yet (they are needed for scene-restricted training in a v2).

In [ ]:
BASE_DIR = (
    "/kaggle/input/datasets/vedantbarhate007/"
    "rsicd-eric/RSICD_optimal-master"
)

IMG_DIR = os.path.join(BASE_DIR, "RSICD_images")
RSICD_JSON = os.path.join(BASE_DIR, "dataset_rsicd.json")

QWEN_DIR = "/kaggle/input/datasets/vedantbarhate007/eric-qwenlabels"

VOCAB_DIR = (
    "/kaggle/input/datasets/vedantbarhate007/"
    "rsicd-raster-classes/raster_classes_dataset"
)

GLOBAL_VOCAB_FILE = os.path.join(VOCAB_DIR, "global_vocab.json")
CLASS_TO_VOCAB_FILE = os.path.join(
    VOCAB_DIR, "class_to_vocab_with_indices.json"
)

OUTPUT_DIR = "/kaggle/working/ERIC_Module5"
os.makedirs(OUTPUT_DIR, exist_ok=True)

required_paths = [
    IMG_DIR,
    RSICD_JSON,
    QWEN_DIR,
    GLOBAL_VOCAB_FILE,
    CLASS_TO_VOCAB_FILE,
]

for path in required_paths:
    print("OK     " if os.path.exists(path) else "MISSING", path)

assert all(os.path.exists(p) for p in required_paths), (
    "A required path is missing. Check Kaggle's Input panel."
)

## 4. RSICD Metadata and Official Splits

`dataset_rsicd.json` carries the official `train` / `val` / `test` assignment per image. These splits are used as-is so that nothing the raster model trains on leaks into the later captioning evaluation. The assertions confirm the three splits are non-empty and disjoint.

In [ ]:
with open(RSICD_JSON, "r", encoding="utf-8") as f:
    rsicd_data = json.load(f)

print("JSON root keys:", list(rsicd_data.keys()))

images_metadata = rsicd_data["images"]

filename_metadata = {
    record["filename"]: record
    for record in images_metadata
}

print("Images in metadata:", len(filename_metadata))
print("Example record:", images_metadata[0])

In [ ]:
split_to_filenames = {"train": [], "val": [], "test": []}

for record in images_metadata:
    filename = record["filename"]
    split = str(record.get("split", "")).lower().strip()

    if split in split_to_filenames:
        split_to_filenames[split].append(filename)

for split, names in split_to_filenames.items():
    print(f"{split:>5}: {len(names):,} images")

assert all(split_to_filenames.values()), (
    "One or more official splits are empty. Inspect the split field."
)

assert len(set(split_to_filenames["train"]) & set(split_to_filenames["val"])) == 0
assert len(set(split_to_filenames["train"]) & set(split_to_filenames["test"])) == 0
assert len(set(split_to_filenames["val"]) & set(split_to_filenames["test"])) == 0

## 5. Load the Verified Raster Labels

One JSON per scene class, each mapping `filename -> {"cells": [{"cell_id", "label"}, ...], "needs_review": bool}`.

Per image the loader:

1. **Validates the grid** — exactly cell IDs `0..48`, non-empty string labels; malformed entries are skipped.
2. **Sorts cells by `cell_id`** so that `labels[i]` is always patch `i` (row-major). Stored order is not guaranteed to be `0..48`; without sorting, labels could be silently misaligned with the patches. This is a no-op for files that are already ordered, and the number of re-sorted entries is reported.
3. **Drops uncorrected placeholder grids** (flagged `needs_review` *and* one repeated label) when `DROP_UNCORRECTED_PLACEHOLDERS` is on.
4. **Raises on duplicates** — usually means raw and corrected files are both in `QWEN_DIR`.

In [ ]:
label_files = sorted(
    glob.glob(os.path.join(QWEN_DIR, "**", "*.json"), recursive=True)
)

print("JSON files found:", len(label_files))
for p in label_files:
    print(os.path.relpath(p, QWEN_DIR))

assert label_files, "No label JSON files found. Check QWEN_DIR."

In [ ]:
raster_by_filename = {}
scene_by_filename = {}   # not used for training yet; kept for per-scene analysis / scene-restricted v2

needs_review_flagged = 0
dropped_placeholders = []
out_of_order = 0

for path in label_files:
    scene_name = Path(path).stem.lower().replace("_corrected", "")

    with open(path, "r", encoding="utf-8") as f:
        file_data = json.load(f)

    for filename, entry in file_data.items():
        if filename in raster_by_filename:
            raise ValueError(
                f"Duplicate raster entry found for {filename}. "
                "QWEN_DIR probably contains both raw and corrected files; "
                "keep only the verified ones."
            )

        cells = entry.get("cells", [])
        stored_ids = [c.get("cell_id", -1) for c in cells]

        # Validate the expected 7x7 grid.
        if sorted(stored_ids) != list(range(49)):
            print(f"Skipping malformed grid: {filename}; found {len(cells)} cells")
            continue

        # Patch i must correspond to cell_id i, whatever the stored order was.
        if stored_ids != list(range(49)):
            out_of_order += 1
            cells = sorted(cells, key=lambda c: c["cell_id"])

        labels = [c.get("label") for c in cells]

        if any(not isinstance(x, str) or not x.strip() for x in labels):
            print("Skipping invalid labels:", filename)
            continue

        needs_review = bool(entry.get("needs_review", False))
        needs_review_flagged += int(needs_review)

        if DROP_UNCORRECTED_PLACEHOLDERS and needs_review and len(set(labels)) == 1:
            dropped_placeholders.append(filename)
            continue

        raster_by_filename[filename] = labels
        scene_by_filename[filename] = scene_name

print("Valid raster images:", len(raster_by_filename))
print("Images per scene:", dict(Counter(scene_by_filename.values())))
print("Unique raster labels:",
      len(set(label for labels in raster_by_filename.values() for label in labels)))
print("Entries flagged needs_review:", needs_review_flagged)
print("Dropped uncorrected placeholder grids:", len(dropped_placeholders))
print("Entries stored out of cell_id order (re-sorted):", out_of_order)

## 6. Usable Images per Split

An image is usable only if it has a valid raster **and** its file exists in `IMG_DIR`. Only the 8 labeled scene classes are present, so these counts are a subset of the official split sizes.

In [ ]:
usable_splits = {}

for split, filenames in split_to_filenames.items():
    usable = [
        name for name in filenames
        if name in raster_by_filename
        and os.path.isfile(os.path.join(IMG_DIR, name))
    ]
    usable_splits[split] = usable

    print(
        f"{split}: {len(usable):,} usable images / "
        f"{len(filenames):,} official images"
    )

for split, names in usable_splits.items():
    assert names, f"No usable images for {split}."

## 7. Label Vocabulary (train split only)

The classifier's classes are exactly the labels that occur in the **train** split. Labels that appear only in val/test cannot be learned and get target `-1` (ignored in loss and metrics). The vocabulary is saved so inference uses the identical index mapping.

In [ ]:
all_training_labels = [
    label
    for filename in usable_splits["train"]
    for label in raster_by_filename[filename]
]

label_counts = Counter(all_training_labels)

label_vocab = sorted(label_counts.keys())
label_to_id = {label: i for i, label in enumerate(label_vocab)}

print("Trainable labels:", len(label_vocab))
print("Most common:", label_counts.most_common(15))
print("Least common:", label_counts.most_common()[-15:])

with open(os.path.join(OUTPUT_DIR, "label_vocab.json"), "w") as f:
    json.dump(label_vocab, f, indent=2)

## 8. Load RemoteCLIP (ViT-B/32)

Weights come from the `chendelong/RemoteCLIP` Hugging Face repo, loaded into the standard `open_clip` ViT-B-32 architecture.

In [ ]:
model_name = "ViT-B-32"

ckpt_path = hf_hub_download(
    "chendelong/RemoteCLIP",
    "RemoteCLIP-ViT-B-32.pt"
)

model, _, preprocess = open_clip.create_model_and_transforms(model_name)
tokenizer = open_clip.get_tokenizer(model_name)

checkpoint = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(checkpoint)

model = model.to(device)

print("RemoteCLIP loaded.")
print("Vision transformer blocks:", len(model.visual.transformer.resblocks))
print("Embedding dimension:", model.visual.proj.shape[-1])

## 9. Text Prototypes (prompt ensembling)

Each label is embedded with 4 prompt templates; the 4 embeddings are averaged and re-normalised into one prototype per class. The prototypes are computed once, detached, and stay **frozen** — only the visual encoder is trained to match them.

In [ ]:
prompt_templates = [
    "a satellite photo of {}",
    "an aerial view of {}",
    "a remote sensing image of {}",
    "a satellite image showing {}",
]

model.eval()

with torch.no_grad():
    text_features = []

    for label in label_vocab:
        prompts = [template.format(label) for template in prompt_templates]

        tokens = tokenizer(prompts).to(device)
        features = model.encode_text(tokens).float()
        features = F.normalize(features, dim=-1)

        feature = F.normalize(features.mean(dim=0), dim=0)
        text_features.append(feature)

    text_features = torch.stack(text_features)

text_features = text_features.detach()

print("Text prototype matrix:", text_features.shape)

## 10. What Gets Trained

Everything is frozen, then the last `NUM_TRAINABLE_BLOCKS` blocks of the **visual** transformer are unfrozen. Note that `ln_post` and `visual.proj` (which sit *after* the blocks) and the whole text tower remain frozen. Optimizer: AdamW with a low learning rate to avoid destroying the pretrained alignment.

In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = False

vision_blocks = model.visual.transformer.resblocks

for block in vision_blocks[-NUM_TRAINABLE_BLOCKS:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

trainable_parameters = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad
]

print("Trainable parameters:", sum(p.numel() for p in trainable_parameters))
print("Total parameters:    ", sum(p.numel() for p in model.parameters()))

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

## 11. Patch Feature Extractor

Runs the **standard** full ViT forward (all blocks, including the last block's attention + MLP — *not* the MaskCLIP shortcut used for the Module 3 zero-shot labeling), then:

1. drops the CLS token, keeping the 49 patch tokens,
2. applies `ln_post` and the projection `proj` into the 512-d CLIP space,
3. L2-normalises each patch vector.

Output: `[B, 49, 512]`. Patch `i` corresponds to grid cell `i` (row-major), matching the label order.

The same function is used for training, validation and testing, so train and inference paths cannot diverge. It also handles both `open_clip` transformer layouts: newer versions are batch-first (`batch_first=True`); older ones expect `[tokens, batch, dim]`.

In [ ]:
def extract_trainable_patch_features(images):
    visual = model.visual

    # images: [B, 3, 224, 224]
    x = visual.conv1(images)                                   # [B, width, 7, 7]

    batch_size, channels, grid_h, grid_w = x.shape

    x = x.reshape(batch_size, channels, grid_h * grid_w).permute(0, 2, 1)

    cls_token = visual.class_embedding.to(x.dtype).reshape(1, 1, -1).expand(batch_size, 1, -1)

    x = torch.cat([cls_token, x], dim=1)                       # [B, 1 + 49, width]
    x = x + visual.positional_embedding.to(x.dtype)
    x = visual.ln_pre(x)

    # Handle both open_clip transformer layouts.
    batch_first = getattr(visual.transformer, "batch_first", False)
    if not batch_first:
        x = x.permute(1, 0, 2)
    x = visual.transformer(x)
    if not batch_first:
        x = x.permute(1, 0, 2)

    patch_tokens = x[:, 1:, :]                                 # drop CLS
    patch_tokens = visual.ln_post(patch_tokens)
    patch_tokens = patch_tokens @ visual.proj                  # [B, 49, 512]

    patch_tokens = F.normalize(patch_tokens.float(), dim=-1)

    return patch_tokens.reshape(batch_size, grid_h * grid_w, -1)

## 12. Dataset and DataLoaders

Each sample is `(image_tensor [3,224,224], target [49], filename)` where `target[i]` is the class id of patch `i`, or `-1` if that label is not in the train vocabulary. Images use RemoteCLIP's own pretrained preprocessing. **No augmentation** is applied (see *Known limitations*).

In [ ]:
class RasterPatchDataset(Dataset):
    def __init__(self, filenames):
        self.filenames = filenames

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        filename = self.filenames[index]

        image = Image.open(os.path.join(IMG_DIR, filename)).convert("RGB")
        image_tensor = preprocess(image)

        labels = raster_by_filename[filename]
        target = torch.tensor(
            [label_to_id.get(label, -1) for label in labels],
            dtype=torch.long
        )

        return image_tensor, target, filename

In [ ]:
train_dataset = RasterPatchDataset(usable_splits["train"])
val_dataset = RasterPatchDataset(usable_splits["val"])
test_dataset = RasterPatchDataset(usable_splits["test"])

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

## 13. Sanity Check — Shapes and Gradient Flow

Before training: confirm features are `[B, 49, 512]`, a loss can be computed, and **gradients actually reach the unfrozen blocks** (catches a silent "nothing is training" failure).

In [ ]:
model.train()

images, targets, filenames = next(iter(train_loader))
images = images[:2].to(device)
targets = targets[:2].to(device)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
    enabled=(device.type == "cuda")
):
    patch_features = extract_trainable_patch_features(images)

print("Input images:", images.shape)
print("Patch features:", patch_features.shape)
print("Patch targets:", targets.shape)

assert patch_features.shape == (2, 49, 512), (
    f"Unexpected feature shape: {patch_features.shape}"
)

optimizer.zero_grad(set_to_none=True)

test_logits = patch_features @ text_features.T
test_loss = F.cross_entropy(
    test_logits.reshape(-1, len(label_vocab)),
    targets.reshape(-1),
    ignore_index=-1
)

test_loss.backward()

gradient_sum = sum(
    p.grad.abs().sum().item()
    for p in trainable_parameters
    if p.grad is not None
)

print("Test loss:", test_loss.item())
print("Gradient sum:", gradient_sum)

assert gradient_sum > 0, "No gradients reached the trainable parameters."

optimizer.zero_grad(set_to_none=True)
model.eval()

print("Sanity check passed.")

## 14. Class Weights and Loss

Patch labels are heavily imbalanced (a few terms like `grass` / `building` / `road` dominate). Each class is weighted by `1/sqrt(train frequency)` (normalised to mean 1) — softer than full inverse-frequency weighting.

The loss is per-patch cross-entropy over **all** vocabulary classes (no scene restriction), ignoring patches with target `-1`.

In [ ]:
counts = torch.tensor(
    [label_counts[label] for label in label_vocab],
    dtype=torch.float32
)

class_weights = 1.0 / torch.sqrt(counts.clamp_min(1))
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device)

print("Number of classes:", len(label_vocab))
print("Weight range:", class_weights.min().item(), class_weights.max().item())

In [ ]:
def patch_logits(patch_features):
    """Cosine similarity to the text prototypes, scaled. -> [B, 49, num_classes]"""
    return (patch_features @ text_features.T) * LOGIT_SCALE


def patch_classification_loss(patch_features, targets):
    """
    patch_features: [B, 49, 512]
    targets:        [B, 49], with -1 for unknown labels
    """
    logits = patch_logits(patch_features)

    num_classes = logits.shape[-1]
    logits = logits.reshape(-1, num_classes)
    targets = targets.reshape(-1)

    valid = targets >= 0

    if not valid.any():
        return logits.sum() * 0.0

    return F.cross_entropy(logits[valid], targets[valid], weight=class_weights)

## 15. Evaluation

`run_inference` collects predictions over a loader; `compute_metrics` turns them into numbers. One definition is used everywhere (validation, baseline, final test report):

| Metric | Meaning |
|---|---|
| `accuracy` | fraction of labeled patches predicted correctly (dominated by frequent classes) |
| `balanced_accuracy` | mean per-class recall over classes present in the ground truth |
| `macro_f1` | macro-F1 over classes that appear in the ground truth **or** the predictions (sklearn default). **Used for model selection.** |
| `macro_f1_all_classes` | macro-F1 over *every* vocabulary class; classes with no support count as F1 = 0, so this is deflated whenever some classes are absent from the split. Reported for comparability with earlier runs only. |

> The original notebook used the `all_classes` definition during training/validation but the `macro_f1` definition in the final test report, so numbers from the two places were not comparable. They are now computed identically.

In [ ]:
@torch.no_grad()
def run_inference(loader):
    model.eval()

    y_true, y_pred = [], []
    total_loss, batches = 0.0, 0

    for images, targets, filenames in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            features = extract_trainable_patch_features(images)

        loss = patch_classification_loss(features, targets)
        total_loss += loss.item()
        batches += 1

        predictions = patch_logits(features).argmax(dim=-1)
        valid = targets >= 0

        y_true.extend(targets[valid].cpu().tolist())
        y_pred.extend(predictions[valid].cpu().tolist())

    return y_true, y_pred, total_loss / max(batches, 1)


def compute_metrics(y_true, y_pred):
    all_labels = list(range(len(label_vocab)))
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1_all_classes": f1_score(
            y_true, y_pred, labels=all_labels, average="macro", zero_division=0
        ),
    }


def evaluate(loader):
    y_true, y_pred, loss = run_inference(loader)

    if not y_true:
        return {
            "loss": float("nan"),
            "accuracy": float("nan"),
            "balanced_accuracy": float("nan"),
            "macro_f1": float("nan"),
            "macro_f1_all_classes": float("nan"),
        }

    return {"loss": loss, **compute_metrics(y_true, y_pred)}

## 16. Baseline — Pretrained RemoteCLIP Before Fine-Tuning

Evaluates the **untouched** pretrained weights with the exact same extractor, prototypes and metrics. This is the reference for "how much did fine-tuning help".

> This baseline uses the standard patch-token path against the full vocabulary. It is **not** the MaskCLIP-style, scene-restricted zero-shot labeling from Module 3, so do not compare it directly to the ~30-40% estimated by eye there.

In [ ]:
baseline_val = evaluate(val_loader)
baseline_test = evaluate(test_loader)

print("Baseline (pretrained, no fine-tuning)")
print("  val :", {k: round(v, 4) for k, v in baseline_val.items()})
print("  test:", {k: round(v, 4) for k, v in baseline_test.items()})

with open(os.path.join(OUTPUT_DIR, "baseline_metrics.json"), "w") as f:
    json.dump({"val": baseline_val, "test": baseline_test}, f, indent=2)

## 17. Training

- Mixed precision (AMP) with gradient scaling, gradient clipping at `GRAD_CLIP_NORM`.
- Constant learning rate (no warm-up or scheduler).
- After every epoch: validation; the checkpoint with the **best validation macro-F1** is saved (full model `state_dict` + vocabulary + metrics).
- The run configuration is saved to `config.json`; per-epoch metrics to `training_history.json`.

In [ ]:
checkpoint_path = os.path.join(
    OUTPUT_DIR, f"remoteclip_last{NUM_TRAINABLE_BLOCKS}blocks_best.pt"
)

config = {
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "num_trainable_blocks": NUM_TRAINABLE_BLOCKS,
    "logit_scale": LOGIT_SCALE,
    "drop_uncorrected_placeholders": DROP_UNCORRECTED_PLACEHOLDERS,
    "num_classes": len(label_vocab),
    "train_images": len(usable_splits["train"]),
    "val_images": len(usable_splits["val"]),
    "test_images": len(usable_splits["test"]),
}
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

best_val_f1 = -1.0
history = []

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

for epoch in range(EPOCHS):
    model.train()

    running_loss = 0.0
    start_time = time.time()

    for step, (images, targets, filenames) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            features = extract_trainable_patch_features(images)
            loss = patch_classification_loss(features, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=GRAD_CLIP_NORM)

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        if step % 50 == 0:
            print(
                f"Epoch {epoch+1}/{EPOCHS} | "
                f"Batch {step}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    train_loss = running_loss / max(len(train_loader), 1)
    val_metrics = evaluate(val_loader)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        **{f"val_{k}": v for k, v in val_metrics.items()}
    })

    print(
        f"\nEpoch {epoch+1}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_metrics['loss']:.4f}, "
        f"val_accuracy={val_metrics['accuracy']:.4f}, "
        f"val_macro_f1={val_metrics['macro_f1']:.4f}, "
        f"time={time.time()-start_time:.1f}s\n"
    )

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]

        torch.save({
            "model_state_dict": model.state_dict(),
            "label_vocab": label_vocab,
            "label_to_id": label_to_id,
            "epoch": epoch + 1,
            "val_metrics": val_metrics,
        }, checkpoint_path)

        print("Saved new best checkpoint:", checkpoint_path)

with open(os.path.join(OUTPUT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)

print("Training complete.")
print("Best validation macro-F1:", best_val_f1)

## 18. Final Evaluation on the Test Split

Restores the best-validation checkpoint and evaluates on the official **test** split exactly once. The comparison against the Section 16 baseline shows the effect of fine-tuning.

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

model.load_state_dict(checkpoint["model_state_dict"])

print("Best epoch:", checkpoint["epoch"])
print("Selected validation metrics:", checkpoint["val_metrics"])

test_metrics = evaluate(test_loader)
print("Final test metrics:     ", {k: round(v, 4) for k, v in test_metrics.items()})
print("Baseline test metrics:  ", {k: round(v, 4) for k, v in baseline_test.items()})

with open(os.path.join(OUTPUT_DIR, "test_metrics.json"), "w") as f:
    json.dump({"fine_tuned": test_metrics, "baseline": baseline_test,
               "best_epoch": checkpoint["epoch"]}, f, indent=2)

## 19. Per-Class Test Report

Precision / recall / F1 / support for every class that appears in the test ground truth **or** predictions. Look here for which classes carry the error — classes with high support but low recall are the real problem, and classes with support 0 but non-zero predictions are over-predicted "attractor" classes.

In [ ]:
y_true, y_pred, _ = run_inference(test_loader)

metrics = compute_metrics(y_true, y_pred)
for name, value in metrics.items():
    print(f"{name:>22}: {value:.4f}")

present = sorted(set(y_true) | set(y_pred))
report_text = classification_report(
    y_true, y_pred,
    labels=present,
    target_names=[label_vocab[i] for i in present],
    zero_division=0
)
print(report_text)

report_dict = classification_report(
    y_true, y_pred,
    labels=present,
    target_names=[label_vocab[i] for i in present],
    zero_division=0,
    output_dict=True
)
with open(os.path.join(OUTPUT_DIR, "per_class_report.json"), "w") as f:
    json.dump(report_dict, f, indent=2)

In [ ]:
for name in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, name)) / 1e6
    print(f"{name:45s} {size_mb:8.2f} MB")

## Known Limitations and v2 Directions

Things this notebook deliberately does **not** do yet, roughly in order of expected impact:

1. **Fixed, low logit scale (`LOGIT_SCALE = 10`).** CLIP-style similarities are normally scaled by ~50-100 (or a learned scale). At 10 the softmax over a ~100-class vocabulary is very soft, which weakens gradients and caps confidence. Try 50-100 or a learnable scale.
2. **Text prototypes are frozen.** Only the visual blocks adapt. Making the prototypes learnable (initialised from the text embeddings) gives the classifier far more freedom; `ln_post` / `proj` are also frozen.
3. **No scene restriction.** Loss and metrics run over the full trained vocabulary, although each image only uses its own scene's candidate terms (~15). Masking logits to the scene's candidates matches the labeling pipeline (`scene_by_filename` and `CLASS_TO_VOCAB_FILE` are already available for this). Numbers from a scene-restricted evaluation are **not** comparable to this notebook's unrestricted numbers.
4. **No augmentation.** Flips and 90-degree rotations are label-preserving if the 7x7 label grid is permuted identically — up to 8x more training data for free. Avoid random crops/scaling (they break patch alignment).
5. **Constant LR, single seed, no scheduler/warm-up.**
6. **Ceiling effects.** 32px patches are inherently ambiguous (grass vs meadow, building vs roof), and labels come from a VLM plus manual correction. Inspect the per-class report and a confusion matrix before assuming the encoder is the bottleneck.